<a href="https://colab.research.google.com/github/fergogu27-ctrl/EDPII/blob/main/Matriz_Fundamental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matriz Fundamental

 **Cadenas absorbentes a tiempo discreto**.

La idea central es ordenar los estados como:


$$\text{transitorios primero, absorbentes después.}$$


Entonces la matriz de transición se escribe en forma canónica:

$$
P=
\begin{pmatrix}
Q & R\\
0 & I
\end{pmatrix}
$$

Donde:

- $Q$ contiene las probabilidades de pasar de estado transitorio a estado transitorio.
- $R$ contiene las probabilidades de pasar de estado transitorio a estado absorbente.
- $I$ corresponde a los estados absorbentes.
- $0$ indica que desde un estado absorbente no se regresa a estados transitorios.

La **matriz fundamental** es:

$$F=(I-Q)^{-1}$$

Si $a_i$ representa el tiempo esperado de absorción iniciando en el estado transitorio $i$, entonces:

$$a_i=\sum_{k\in T}F_{ik}$$

En forma matricial:

$$a=F\mathbf{1}$$

Además, las probabilidades de absorción se calculan con:

$$B=FR.$$

Cada entrada $B_{ij}$ representa la probabilidad de terminar en el estado absorbente $j$, dado que se inició en el estado transitorio $i$.

# Ejercicio 1. Promedio de tiradas para terminar serpientes y escaleras

## Interpretación del tablero

Tomamos como estados las casillas:

$$S=\{0,1,2,\dots,20\}$$

La casilla $0$ representa el inicio, antes de la casilla 1.

La casilla $20$ es el estado absorbente, porque al llegar ahí termina el juego.

Por tanto:


$$T=\{0,1,2,\dots,19\},
\qquad
A=\{20\}$$

De la imagen observamos las siguientes serpientes y escaleras:

$$
3\to 11,
\qquad
15\to 19,
\qquad
13\to 5,
\qquad
17\to 9.
$$

Usaremos la regla usual de que para terminar se debe llegar exactamente a la casilla 20.  
Si una tirada rebasa 20, el jugador permanece en la misma casilla y la tirada sí cuenta.

In [1]:
import numpy as np

N = 20          # Estado absorbente final.

saltos = {
    3: 11,    # escalera
    15: 19,   # escalera
    13: 5,    # serpiente
    17: 9     # serpiente
}

def mover(posicion, dado):
    nueva = posicion + dado       # Primero se avanza según el número obtenido en el dado
    if nueva > N:                 # Si se rebasa 20, el jugador no se mueve.
        return posicion

    return saltos.get(nueva, nueva)

## Construcción de la matriz de transición de serpientes y escaleras

Como el dado es justo,

$$P(X_{n+1}=j\mid X_n=i)
=
\frac{\#\{\text{caras del dado que llevan de }i\text{ a }j\}}{6}.$$

En este problema hay 20 estados transitorios y 1 estado absorbente.  
Por eso:


$$Q \in \mathbb{R}^{20\times 20},
\qquad
R\in \mathbb{R}^{20\times 1}.$$

In [2]:
num_estados = N + 1       # Número de estados totales: 0,1,2,...,20

P = np.zeros((num_estados, num_estados))    # Creamos la matriz P completa de tamaño 21 x 21.

for i in range(N):
    for dado in range(1, 7):
        j = mover(i, dado)
        P[i, j] += 1/6         # Como cada cara tiene probabilidad 1/6, sumamos 1/6.
P[N, N] = 1

transitorios = list(range(0, 20))         # Estados transitorios y absorbentes.
absorbentes = [20]

Q = P[np.ix_(transitorios, transitorios)]
R = P[np.ix_(transitorios, absorbentes)]

print("Dimensión de Q:", Q.shape)
print("Dimensión de R:", R.shape)
print("La matriz P queda en forma canónica porque usamos transitorios primero y absorbentes al final.")

Dimensión de Q: (20, 20)
Dimensión de R: (20, 1)
La matriz P queda en forma canónica porque usamos transitorios primero y absorbentes al final.


## Matriz fundamental y tiempo esperado de absorción

Ahora usamos :

$$F=(I-Q)^{-1}$$

Luego, el tiempo esperado de absorción desde cada estado transitorio se obtiene sumando los renglones de $F$:

$$a_i=\sum_{k\in T}F_{ik}$$

En forma de vector:

$$a=F\mathbf{1}$$

In [3]:
I = np.eye(Q.shape[0])    # Matriz identidad del mismo tamaño que Q.

F = np.linalg.inv(I - Q)      # Calculamos la matriz fundamental.

unos = np.ones((Q.shape[0], 1))

a = F @ unos
promedio_tiradas = a[0, 0]      # El estado inicial es 0.


print("Tiempos esperados de absorción:")
for estado, valor in zip(transitorios, a.flatten()):
    print(f"a_{estado} = {valor:.6f}")

print()
print(f"Promedio de tiradas iniciando en 0: {promedio_tiradas:.6f}")

Tiempos esperados de absorción:
a_0 = 11.595200
a_1 = 11.424228
a_2 = 11.202016
a_3 = 11.252773
a_4 = 10.925344
a_5 = 10.651943
a_6 = 10.356138
a_7 = 10.398396
a_8 = 9.868746
a_9 = 9.316068
a_10 = 8.960775
a_11 = 9.011531
a_12 = 8.581313
a_13 = 7.805989
a_14 = 6.690848
a_15 = 6.829017
a_16 = 6.829017
a_17 = 6.000000
a_18 = 6.000000
a_19 = 6.000000

Promedio de tiradas iniciando en 0: 11.595200


In [4]:
F         #Matriz F

array([[1.        , 0.16666667, 0.19444444, 0.        , 0.22685185,
        0.74379623, 0.38862653, 0.28673095, 0.30674167, 0.67660459,
        0.4382253 , 0.70030607, 0.46620585, 0.        , 0.43134725,
        0.        , 0.50902112, 0.        , 0.70328711, 4.35634453],
       [0.        , 1.        , 0.16666667, 0.        , 0.19444444,
        0.7252153 , 0.34772107, 0.40567458, 0.30662034, 0.67894729,
        0.44310384, 0.67899151, 0.4768431 , 0.        , 0.43075101,
        0.        , 0.50742237, 0.        , 0.70750824, 4.35431838],
       [0.        , 0.        , 1.        , 0.        , 0.16666667,
        0.69873097, 0.31089961, 0.36271621, 0.42316891, 0.67277351,
        0.43915931, 0.65124142, 0.47665983, 0.        , 0.44383383,
        0.        , 0.5027236 , 0.        , 0.71160863, 4.34183395],
       [0.        , 0.        , 0.        , 1.        , 0.16666667,
        0.70581323, 0.31207998, 0.36409331, 0.42477553, 0.81668251,
        0.46501854, 0.51474385, 0.48289895, 0

Por lo tanto, analíticamente:

$$\boxed{a_0\approx 11.5952}$$

Es decir, el número promedio de tiradas necesarias para terminar el juego es aproximadamente:

$$\boxed{11.5952\text{ tiradas}}$$

## Simulación Monte Carlo para serpientes y escaleras

La simulación repite muchas partidas completas.

En cada partida:

1. Se empieza en la casilla 0.
2. Se lanza el dado.
3. Se avanza.
4. Si cae en serpiente o escalera, se aplica el salto.
5. Si rebasa 20, permanece en la misma casilla.
6. Se cuenta el número de tiradas hasta llegar a 20.

In [5]:
def simular_P(rng):
    posicion = 0
    tiradas = 0
    while posicion != N:          # Mientras no llegue a la casilla final, sigue jugando.
        dado = rng.integers(1, 7)
        posicion = mover(posicion, dado)
        tiradas += 1

    return tiradas


def simular_serpientes(num_simulaciones=100000, semilla=123):
    rng = np.random.default_rng(semilla)
    resultados = np.array([
        simular_P(rng)
        for _ in range(num_simulaciones)
    ])
    media = resultados.mean()
    desviacion = resultados.std(ddof=1)
    error_estandar = desviacion / np.sqrt(num_simulaciones)
    li = media - 1.96 * error_estandar
    ls = media + 1.96 * error_estandar

    return media, li, ls


media, li, ls = simular_serpientes()

print(f"Promedio simulado = {media:.6f}")
print(f"Intervalo de confianza 95% = ({li:.6f}, {ls:.6f})")

Promedio simulado = 11.650550
Intervalo de confianza 95% = (11.610742, 11.690358)


# Ejercicio 2. Probabilidad de que el ratón alcance la comida

Ahora usamos la misma idea de cadena absorbente.

Los estados son:

$$S=\{0,1,2,3,4,5,6,7,8\}$$

Los estados absorbentes son:

$$A=\{7,8\}$$

Donde:

- $7$ representa comida.
- $8$ representa shock.

Los estados transitorios son:

$$T=\{0,1,2,3,4,5,6\}$$

Leyendo los caminos de la imagen, usamos los vecinos:


$$\begin{aligned}
V(0)&=\{1,2\},\\
V(1)&=\{0,3,7\},\\
V(2)&=\{0,3,8\},\\
V(3)&=\{1,2,4,5\},\\
V(4)&=\{3,6,7\},\\
V(5)&=\{3,6,8\},\\
V(6)&=\{4,5\}.
\end{aligned}$$

En cada casilla transitoria el ratón elige con igual probabilidad uno de los caminos disponibles.

## Matrices

Como los estados transitorios son $0,1,2,3,4,5,6$ , la matriz $Q$ es de tamaño $7\times 7$

Tomando las columnas también en el orden $0,1,2,3,4,5,6$:


$$Q=
\begin{pmatrix}
0 & 1/2 & 1/2 & 0 & 0 & 0 & 0\\
1/3 & 0 & 0 & 1/3 & 0 & 0 & 0\\
1/3 & 0 & 0 & 1/3 & 0 & 0 & 0\\
0 & 1/4 & 1/4 & 0 & 1/4 & 1/4 & 0\\
0 & 0 & 0 & 1/3 & 0 & 0 & 1/3\\
0 & 0 & 0 & 1/3 & 0 & 0 & 1/3\\
0 & 0 & 0 & 0 & 1/2 & 1/2 & 0
\end{pmatrix}.$$

Como los estados absorbentes son $7$ y $8$ , la matriz $R$ tiene dos columnas:

$$R=
\begin{pmatrix}
0 & 0\\
1/3 & 0\\
0 & 1/3\\
0 & 0\\
1/3 & 0\\
0 & 1/3\\
0 & 0
\end{pmatrix}$$

La primera columna corresponde a la comida y la segunda columna corresponde al shock.

In [6]:
vecinos = {
    0: [1, 2],
    1: [0, 3, 7],
    2: [0, 3, 8],
    3: [1, 2, 4, 5],
    4: [3, 6, 7],
    5: [3, 6, 8],
    6: [4, 5]
}

T_raton = [0, 1, 2, 3, 4, 5, 6]     # Estados transitorios y absorbentes.
A_raton = [7, 8]

indice_T = {estado: i for i, estado in enumerate(T_raton)}
indice_A = {estado: i for i, estado in enumerate(A_raton)}

Q_raton = np.zeros((len(T_raton), len(T_raton)))
R_raton = np.zeros((len(T_raton), len(A_raton)))

for estado in T_raton:

    grado = len(vecinos[estado])
    prob = 1 / grado
    for vecino in vecinos[estado]:
        if vecino in indice_T:
            Q_raton[indice_T[estado], indice_T[vecino]] += prob
        if vecino in indice_A:
            R_raton[indice_T[estado], indice_A[vecino]] += prob

print("Q del ratón:")
print(Q_raton)

print()
print("R del ratón:")
print(R_raton)

Q del ratón:
[[0.         0.5        0.5        0.         0.         0.
  0.        ]
 [0.33333333 0.         0.         0.33333333 0.         0.
  0.        ]
 [0.33333333 0.         0.         0.33333333 0.         0.
  0.        ]
 [0.         0.25       0.25       0.         0.25       0.25
  0.        ]
 [0.         0.         0.         0.33333333 0.         0.
  0.33333333]
 [0.         0.         0.         0.33333333 0.         0.
  0.33333333]
 [0.         0.         0.         0.         0.5        0.5
  0.        ]]

R del ratón:
[[0.         0.        ]
 [0.33333333 0.        ]
 [0.         0.33333333]
 [0.         0.        ]
 [0.33333333 0.        ]
 [0.         0.33333333]
 [0.         0.        ]]


## Cálculo de probabilidades de absorción

Por los apuntes, la matriz fundamental es:

$$F=(I-Q)^{-1}$$

Luego, las probabilidades de absorción son:

$$B=FR$$

En este caso:

- La primera columna de $B$ da la probabilidad de llegar a la comida.
- La segunda columna de $B$ da la probabilidad de llegar al shock.

In [7]:
I_raton = np.eye(Q_raton.shape[0])
F_raton = np.linalg.inv(I_raton - Q_raton)        # Matriz fundamental del ratón.

B_raton = F_raton @ R_raton     # Matriz de probabilidades de absorción.
print("Matriz fundamental F:")
print(F_raton)
print()
print("Matriz de probabilidades de absorción B = F R:")
print(B_raton)
print()
print("Probabilidad de llegar a comida iniciando en 0:")
print(B_raton[indice_T[0], indice_A[7]])
print()
print("Probabilidad de llegar a shock iniciando en 0:")
print(B_raton[indice_T[0], indice_A[8]])

Matriz fundamental F:
[[1.75  1.125 1.125 1.    0.375 0.375 0.25 ]
 [0.75  1.625 0.625 1.    0.375 0.375 0.25 ]
 [0.75  0.625 1.625 1.    0.375 0.375 0.25 ]
 [0.5   0.75  0.75  2.    0.75  0.75  0.5  ]
 [0.25  0.375 0.375 1.    1.625 0.625 0.75 ]
 [0.25  0.375 0.375 1.    0.625 1.625 0.75 ]
 [0.25  0.375 0.375 1.    1.125 1.125 1.75 ]]

Matriz de probabilidades de absorción B = F R:
[[0.5        0.5       ]
 [0.66666667 0.33333333]
 [0.33333333 0.66666667]
 [0.5        0.5       ]
 [0.66666667 0.33333333]
 [0.33333333 0.66666667]
 [0.5        0.5       ]]

Probabilidad de llegar a comida iniciando en 0:
0.5

Probabilidad de llegar a shock iniciando en 0:
0.5


Por lo tanto:
$$P(\text{llegar a comida}\mid X_0=0)=B_{0,7}$$

Del cálculo:
$$\boxed{P(\text{comida}\mid X_0=0)=0.5}$$
y
$$\boxed{P(\text{shock}\mid X_0=0)=0.5}$$

Entonces, la probabilidad de que el ratón alcance la comida iniciando en la casilla 0 es:

$$\boxed{0.5=50\%}$$

## Simulación Monte Carlo

En cada simulación:

1. El ratón empieza en la casilla 0.
2. Mientras no llegue a 7 ni a 8, elige aleatoriamente un vecino disponible.
3. Si llega a 7, se cuenta como éxito.
4. Si llega a 8, se cuenta como fracaso.

In [8]:
def simular_recorrido_raton(rng):
    estado = 0

    while estado not in [7, 8]:
        estado = rng.choice(vecinos[estado])
    if estado == 7:
        return 1
    return 0

def simular_raton(num_simulaciones=100000, semilla=123):

    rng = np.random.default_rng(semilla)
    resultados = np.array([
        simular_recorrido_raton(rng)
        for _ in range(num_simulaciones)
    ])

    phat = resultados.mean()
    error_estandar = np.sqrt(phat * (1 - phat) / num_simulaciones)
    li = phat - 1.96 * error_estandar
    ls = phat + 1.96 * error_estandar

    return phat, li, ls


phat, li, ls = simular_raton()

print(f"Probabilidad simulada de llegar a comida = {phat:.6f}")
print(f"Intervalo de confianza 95% = ({li:.6f}, {ls:.6f})")

Probabilidad simulada de llegar a comida = 0.500670
Intervalo de confianza 95% = (0.497571, 0.503769)


# Conclusiones

Siguiendo el método de cadenas absorbentes:


$$P=
\begin{pmatrix}
Q & R\\
0 & I
\end{pmatrix},
\qquad
F=(I-Q)^{-1}$$

Para serpientes y escaleras, usando:

$$a=F\mathbf{1}$$

se obtiene:
$$\boxed{a_0\approx 11.5952}$$

Por lo tanto, el número promedio de tiradas necesarias para terminar el juego es aproximadamente:
$$\boxed{11.5952\text{ tiradas}}$$

Para el ratón, usando:

$$B=FR$$

se obtiene:
$$\boxed{P(\text{comida}\mid X_0=0)=0.5}$$

Por lo tanto, la probabilidad de que el ratón alcance la comida iniciando en la casilla 0 es:
$$\boxed{50\%}$$